The point of this file is to create a notebook that finds 5 markets within the specific timeframe of june 1 2025 - march 1 2026, and acquire all the specific data necessary for k-means testing.

In [60]:
# Imports and setup
import requests
import time
import json
from pprint import pprint
from datetime import datetime, timedelta
import pandas as pd

from py_clob_client_v2 import (
    ClobClient,
    OrderArgs,
    MarketOrderArgs,
    OrderType,
    OpenOrderParams,
    BalanceAllowanceParams,
    AssetType,
    Side,
)

GAMMA_API = "https://gamma-api.polymarket.com"  # Market discovery
DATA_API = "https://data-api.polymarket.com"    # User positions & activity
CLOB_API = "https://clob.polymarket.com"        # Order book & trading
EVENT_SLUG = "israel-military-action-against-iran-before-july"
YES_TOKEN_ID = "109681959945973300464568698402968596289258214226684818748321941747028805721376"


In [41]:

# Market metadata
market = requests.get(f"{GAMMA_API}/markets/slug/{EVENT_SLUG}").json()

# Trade history (reconstructs price over time for resolved markets)
trades = requests.get(f"{DATA_API}/trades", params={
    "market": YES_TOKEN_ID,
    "limit": 500,
}).json()
# Accessing the liquidity key from your existing market metadata response
total_liquidity = float(market.get("liquidity", 0))

# Pull daily structural buckets
volume_history = requests.get(f"{CLOB_API}/volume-history", params={"market": YES_TOKEN_ID}).json()

# Sum up the transaction counts across all historical days
total_trades = sum(int(day.get("count", 0)) for day in volume_history.get("history", []))

print(f"Question: {market.get('question')}")
print(f"Volume:   ${float(market.get('volume', 0)):,.2f}")
print(f"Resolved: {market.get('closed')}")
print(f"Created At:  {market.get('createdAt')}")  # <--- Added key
print(f"Closed Time: {market.get('closedTime')}") # <--- Added key
print(f"Trades returned: {len(trades)}")
print(f"Total Aggregated Liquidity: ${total_liquidity:,.2f}")
print(f"Total Trades: ${total_trades:,.2f}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [61]:
from datetime import datetime

# Timestamps returned from your previous cell
created_at_str = "2025-03-31T16:47:35.478417Z"
closed_time_str = "2025-06-13 03:24:45+00"

# Convert ISO strings to timezone-aware datetime objects
# Replacing "Z" with explicit +00:00 offset ensures compatibility across Python versions
start_dt = datetime.fromisoformat(created_at_str.replace("Z", "+00:00"))
end_dt = start_dt + timedelta(days=5)  # 2. Changed from closed_time_str to start_dt + 5 days

# Convert to exact Unix integer timestamps
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# Define parameters targeting the exact lifecycle of the market
params = {
    "market": YES_TOKEN_ID,
    "startTs": start_ts, 
    "endTs": end_ts,
    "fidelity": 60 
}

history = requests.get(f"{CLOB_API}/prices-history", params=params).json()
prices = history.get("history", [])

print(f"Returned {len(prices)} price data points for Israel/Iran.")
if prices:
    prices.sort(key=lambda x: x["t"])
    print(f"Initial Price: ${prices[0]['p']:.2f}")


Returned 119 price data points for Israel/Iran.
Initial Price: $0.32


In [ ]:
import requests

GAMMA_API = "https://gamma-api.polymarket.com"
EVENT_SLUG = "israel-military-action-against-iran-before-july"

# 1. Fetch the market metadata
market_data = requests.get(f"{GAMMA_API}/markets/slug/{EVENT_SLUG}").json()

# 2. Extract options dynamically using Polymarket's multiple fallback keys
# Binary and scalar markets place sibling choices in different structural blocks
sibling_markets = market_data.get("groupMarkets")

if not sibling_markets and "events" in market_data:
    # Fallback to checking the parent event array if groupMarkets isn't at the root
    events = market_data.get("events", [])
    if events:
        sibling_markets = events[0].get("markets", [])

# 3. Calculate total options and the mathematical even split threshold
num_options = len(sibling_markets) if sibling_markets else 0

if num_options > 0:
    even_split_threshold = 1.0 / num_options
    
    print(f"Question Title:         {market_data.get('question')}")
    print(f"Total Choices/Options:  {num_options}")
    print(f"Even Split Threshold:   ${even_split_threshold:.2f} ({even_split_threshold*100:.1f}%)")
    print("\nAll Available Sibling Options in this Group:")
    for option in sibling_markets:
        # Pull name or group title if available
        title = option.get('groupItemTitle') or option.get('title') or option.get('question')
        print(f" - {title}")
else:
    # If it is a completely isolated standalone binary market with no group siblings
    print(f"Question Title:         {market_data.get('question')}")
    print("Total Choices/Options:  2 (Standard Standalone Yes/No Binary Market)")
    print("Even Split Threshold:   $0.50 (50.0%)")


Question Title:         Israel military action against Iran before July?
Total Choices/Options:  2 (Standard Standalone Yes/No Binary Market)
Even Split Threshold:   $0.50 (50.0%)


In [38]:
# 1. Fetch market metadata to get core fields and conditionId
market = requests.get(f"{GAMMA_API}/markets/slug/{EVENT_SLUG}").json()
condition_id = market.get("conditionId")
total_volume = float(market.get("volume", 0))

# Parse operational lifespan dates
start_dt = datetime.fromisoformat(market.get('createdAt').replace("Z", "+00:00"))
end_dt = datetime.fromisoformat(market.get('closedTime'))
duration_hours = max((end_dt - start_dt).total_seconds() / 3600.0, 1.0)

# 2. Paginate through the real Trade Log to count transactions
total_trades = 0
last_timestamp = int(end_dt.timestamp())
start_timestamp = int(start_dt.timestamp())

print("Counting transaction ledger blocks...")
while last_timestamp > start_timestamp:
    params = {
        "market": condition_id, # Using conditionId isolates this event perfectly
        "limit": 1000,
        "timestamp_less_than": last_timestamp
    }
    
    trades_chunk = requests.get(f"{DATA_API}/trades", params=params).json()
    
    if not trades_chunk:
        break
        
    total_trades += len(trades_chunk)
    
    # Move the pagination window backward to the oldest trade in this chunk
    last_timestamp = int(trades_chunk[-1].get("timestamp"))
    
    # Small sleep to respect rate limits if pulling huge datasets
    time.sleep(0.1)

# 3. Derive K-Means features accurately using real trading records
trading_intensity_per_hour = total_volume / duration_hours
avg_trade_size = total_volume / total_trades if total_trades > 0 else 0

print(f"\nQuestion:                  {market.get('question')}")
print(f"Market Lifespan Duration:  {duration_hours / 24.0:.2f} Days ({duration_hours:.1f} Hours)")
print(f"Total Trades (True Count): {total_trades:,}")
print(f"--------------------------------------------------")
print(f"K-MEANS FEATURE: Trading Intensity: ${trading_intensity_per_hour:,.2f} per hour")
print(f"K-MEANS FEATURE: Avg Trade Size:    ${avg_trade_size:,.2f} per transaction")
print(f"K-MEANS FEATURE: Market Duration:   {duration_hours:.1f} Hours")


Counting transaction ledger blocks...


KeyboardInterrupt: 

In [39]:
import requests

url = "https://clob.polymarket.com/prices-history"
token_id = '21271000291843361249209065706097167029083067325856089903026951915683588703117'

# This works - returns data
params_12h = {
    "market": token_id,
    "interval": "max",
    "fidelity": 60*12  # 12 hours in minutes
}
response = requests.get(url, params=params_12h)
print("12-hour:", response.json())
# Returns: {'history': [{'t': ..., 'p': ...}, ...]} 

# This fails - returns empty
params_1h = {
    "market": token_id,
    "interval": "max",
    "fidelity": 60  # 1 hour in minutes
}
response = requests.get(url, params=params_1h)
print("1-hour:", response.json())
# Returns: {'history': []} 

12-hour: {'history': [{'t': 1704844803, 'p': 0.5}, {'t': 1704888003, 'p': 0.0195}, {'t': 1704931203, 'p': 0.0195}, {'t': 1704974403, 'p': 0.0195}, {'t': 1705017603, 'p': 0.0195}, {'t': 1705060803, 'p': 0.0195}, {'t': 1705104002, 'p': 0.019}, {'t': 1705147202, 'p': 0.0185}, {'t': 1705190402, 'p': 0.0185}, {'t': 1705233602, 'p': 0.0185}, {'t': 1705276803, 'p': 0.0185}, {'t': 1705320002, 'p': 0.0185}, {'t': 1705363202, 'p': 0.0185}, {'t': 1705406402, 'p': 0.0185}, {'t': 1705449603, 'p': 0.0185}, {'t': 1705492802, 'p': 0.0185}, {'t': 1705536002, 'p': 0.0185}, {'t': 1705579202, 'p': 0.0185}, {'t': 1705622402, 'p': 0.0185}, {'t': 1705665602, 'p': 0.0185}, {'t': 1705708803, 'p': 0.021}, {'t': 1705752003, 'p': 0.022}, {'t': 1705795202, 'p': 0.022}, {'t': 1705838403, 'p': 0.022}, {'t': 1705881602, 'p': 0.022}, {'t': 1705924803, 'p': 0.0215}, {'t': 1705968003, 'p': 0.0215}, {'t': 1706011202, 'p': 0.0215}, {'t': 1706054402, 'p': 0.0215}, {'t': 1706097603, 'p': 0.0215}, {'t': 1706140802, 'p': 0.02

In [59]:
# 1. Resolve the correct conditionId via Gamma public lookup
print("Resolving market ID structure...")
market_lookup = requests.get(f"{GAMMA_API}/markets", params={"clobTokenIds": YES_TOKEN_ID}).json()

if isinstance(market_lookup, list) and len(market_lookup) > 0:
    market_data = market_lookup[0]  # <--- CRITICAL FIX: Extract the first dict from the list array
    condition_id = market_data.get("conditionId")
    print(f"Targeting Condition ID: {condition_id}")
else:
    print("Error: Could not resolve this Token ID via the public indexer.")
    condition_id = None

# 2. Paginate through the transaction log using an index offset loop
if condition_id:
    all_trades = []
    current_offset = 0
    chunk_limit = 500  

    print("Downloading transaction ledger blocks...")
    while True:
        params = {
            "market": condition_id, 
            "limit": chunk_limit,
            "offset": current_offset
        }
        response = requests.get(f"{DATA_API}/trades", params=params)
        
        # Check if the server returned a plain text "error" message before parsing JSON
        if response.text.strip() == "error" or not response.text:
            print("Server termination string hit. Ending download loop.")
            break
            
        try:
            trades_chunk = response.json()
        except Exception:
            print("Encountered unparseable payload. Ending download loop.")
            break
        
        if not trades_chunk or len(trades_chunk) == 0:
            break
            
        all_trades.extend(trades_chunk)
        
        if len(trades_chunk) < chunk_limit:
            break
            
        current_offset += chunk_limit
        time.sleep(0.05)

    print(f"Downloaded {len(all_trades):,} clean transaction entries.")

    # 3. Process data directly into a Pandas DataFrame
    clean_trades = [t for t in all_trades if isinstance(t, dict) and "size" in t]

    if clean_trades:
        df = pd.DataFrame(clean_trades)
        
        # Calculate transaction dollar volume (shares * execution price)
        df["trade_volume_usd"] = df["size"].astype(float) * df["price"].astype(float)
        
        # Convert unix timestamp integer to UTC datetime index
        df["datetime"] = pd.to_datetime(df["timestamp"].astype(int), unit="s", utc=True)
        df.set_index("datetime", inplace=True)
        
        # Resample into 1-hour blocks and sum the volume
        hourly_volume = df["trade_volume_usd"].resample("1h").sum().to_frame()
        hourly_volume.rename(columns={"trade_volume_usd": "hourly_volume_usd"}, inplace=True)
        
        # Extract peak metrics for your NB02 K-Means pipeline
        peak_hour_val = hourly_volume["hourly_volume_usd"].max()
        print(f"\nPeak Hourly Trading Intensity: ${peak_hour_val:,.2f}\n")
        
        # Display top 10 most intense trading hours found in the history log
        print("Top 10 Most Intense Trading Hours:")
        print(hourly_volume.sort_values(by="hourly_volume_usd", ascending=False).head(10))
    else:
        print("No valid transactions found for this market selection.")


Resolving market ID structure...
Targeting Condition ID: 0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be
Downloaded 3,501 clean transaction entries.

Peak Hourly Trading Intensity: $4,876.26

Top 10 Most Intense Trading Hours:
                           hourly_volume_usd
datetime                                    
2026-06-06 11:00:00+00:00        4876.264399
2026-06-04 07:00:00+00:00        1257.530000
2026-06-25 23:00:00+00:00         953.505797
2026-06-25 17:00:00+00:00         646.158600
2026-07-03 01:00:00+00:00         588.251200
2026-06-04 06:00:00+00:00         581.651100
2026-06-21 05:00:00+00:00         530.790199
2026-06-05 19:00:00+00:00         476.520000
2026-06-21 10:00:00+00:00         449.000000
2026-06-04 04:00:00+00:00         429.120000


In [56]:
# 1. Run a quick check over the items currently stored in your memory
non_dicts = [item for item in all_trades if not isinstance(item, dict)]

print(f"Total non-dictionary items found: {len(non_dicts)}")
if non_dicts:
    print(f"Sample of a non-dictionary item:\n{non_dicts[0]}")
    print(f"Python data type of this item:  {type(non_dicts[0])}")


Total non-dictionary items found: 1
Sample of a non-dictionary item:
error
Python data type of this item:  <class 'str'>


In [65]:
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import requests

# The official decentralized Subgraph Gateway for Polymarket's CTF Exchange Contract
SUBGRAPH_URL = "https://thegraph.com"

# The specific conditionId you successfully resolved for Israel/Iran
CONDITION_ID = "0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be"

# 1. Structure the GraphQL Query targeting the 'funder' ledger trades
# We filter explicitly by conditionId to retrieve historical 2025 transactions
query = """
{
  transactions(
    first: 1000, 
    where: { market: "%s" }, 
    orderBy: timestamp, 
    orderDirection: asc
  ) {
    id
    timestamp
    amount
    tradeAmount
  }
}
""" % CONDITION_ID

print("Querying decentralized Polymarket Subgraph index...")
response = requests.post(SUBGRAPH_URL, json={"query": query})
data = response.json()

# Extract transaction list array from GraphQL payload nests
records = data.get("data", {}).get("transactions", [])
print(f"Successfully retrieved {len(records):,} historical trade entries from the blockchain.")

# 2. Process data directly into a Pandas DataFrame
if records:
    df = pd.DataFrame(records)
    
    # Map the block timestamps to a clean UTC datetime index
    df["datetime"] = pd.to_datetime(df["timestamp"].astype(int), unit="s", utc=True)
    df.set_index("datetime", inplace=True)
    
    # Use tradeAmount (actual USD swapped) as the volume metric; fallback to raw amount if needed
    df["volume_usd"] = df["tradeAmount"].astype(float)
    
    # 3. Resample the data into clean 1-hour windows for your K-Means features
    hourly_volume = df["volume_usd"].resample("1h").sum().to_frame()
    hourly_volume.rename(columns={"volume_usd": "hourly_volume_usd"}, inplace=True)
    
    # Isolate feature metrics for your model
    peak_intensity = hourly_volume["hourly_volume_usd"].max()
    peak_time = hourly_volume["hourly_volume_usd"].idxmax()
    
    # 4. Render the Volume vs. Time Graph
    plt.figure(figsize=(12, 5))
    plt.plot(hourly_volume.index, hourly_volume["hourly_volume_usd"], color="#e377c2", linewidth=2, label="Historical Volume")
    plt.fill_between(hourly_volume.index, hourly_volume["hourly_volume_usd"], color="#e377c2", alpha=0.15)
    
    plt.scatter(peak_time, peak_intensity, color="red", s=120, zorder=5, label="Peak Intensity Spike")
    
    plt.title(f"Polymarket Subgraph: Historical Hourly Trading Volume vs Time (2025)", fontsize=11, fontweight="bold")
    plt.xlabel("Timeline (2025 UTC Calendar Date)", fontsize=10)
    plt.ylabel("Volume (USD Traded)", fontsize=10)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(loc="upper left")
    plt.tight_layout()
    plt.show()
    
    print(f"K-MEANS FEATURE: True Peak Hourly Intensity: ${peak_intensity:,.2f}")
    print(f"K-MEANS FEATURE: True Average Hourly Intensity: ${hourly_volume['hourly_volume_usd'].mean():,.2f}")
else:
    print("The query executed but no historical records were returned for this market ID.")


Querying decentralized Polymarket Subgraph index...


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [13]:
from dune_client.client import DuneClient
import pandas as pd

DUNE_API_KEY = "ciVRgLegSo4JCLoE4Wzmpt5V2ScN0Tf1"
QUERY_ID = 7930905

dune = DuneClient(DUNE_API_KEY)

all_rows = []
batch_size = 30000
offset = 0

while True:
    try:
        result = dune.get_latest_result(
            QUERY_ID,
            batch_size=batch_size
        )
        rows = result.result.rows
        if not rows:
            break
        all_rows.extend(rows)
        print(f"Fetched {len(all_rows)} rows so far...")
        if len(rows) < batch_size:
            break
        offset += batch_size
    except Exception as e:
        print(f"Stopped at offset {offset}: {e}")
        break

dune_df = pd.DataFrame(all_rows)
print(f"Total rows fetched: {len(dune_df)}")
dune_df.to_parquet("../data/raw/dune_market_trades.parquet", index=False)

Stopped at offset 0: 402 Client Error: Payment Required for url: https://api.dune.com/api/v1/execution/01KXHJVXG6W68DCN3HHAHHQFZT/results?allow_partial_results=true&limit=30000
Total rows fetched: 0
